Поиск по корпусу. Taiga 15–20 МБ.

POS Tags
ADJ – ADP – ADV – AUX – CCONJ – DET – INTJ – NOUN – NUM – PART – PRON – PROPN – PUNCT – SCONJ – SYM – VERB – X

Features
Abbr – Animacy – Aspect – Case – Degree – Foreign – Gender – Mood – NameType – Number – NumForm – NumType – Person – Polarity – 

Poss – PronType – Reflex – Tense – Typo – Variant – VerbForm – Voice

Relations
acl – acl:relcl – advcl – advmod – amod – appos – aux – aux:pass – case – cc – ccomp – compound – conj – cop – csubj – csubj:outer 

– csubj:pass – dep – det – discourse – dislocated – expl – fixed – flat – flat:foreign – flat:name – goeswith – iobj – list – mark – 

nmod – nsubj – nsubj:outer – nsubj:pass – nummod – nummod:entity – nummod:gov – obj – obl – obl:agent – obl:tmod – orphan – parataxis – 

punct – reparandum – root – vocative – xcomp

In [3]:
! pip install conllu requests 

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import requests

url = "https://raw.githubusercontent.com/UniversalDependencies/UD_Russian-Taiga/master/ru_taiga-ud-train.conllu"
file_name = "ru_taiga-ud-train.conllu"


# Скачивание файла
response = requests.get(url) # HTTP зарос 
if response.status_code == 200: # код успешного выполнения 
    with open(file_name, "w", encoding="utf-8") as f:
        f.write(response.text) #Строка f.write(response.text) записывает содержимое текста, полученного в ответе от сервера (response.text), в файл.
    print(f"Файл {file_name} успешно загружен.")
else:
    print(f"Ошибка при загрузке: {response.status_code}")

Файл ru_taiga-ud-train.conllu успешно загружен.


Поиск в корпусе по лемме

In [ ]:
import spacy

In [ ]:
def search_lemma(file, s_lemma, chunk_size=100000, max_chunks=10): # чтобы не занимать много оперативки, обрабатываем по чанкам с размером 100000 символов, и у spacy есть ограничение в 100000 с. 
    #Далее, чтобы не ждать 10 лет, берем толкьо 1 10 чанков из 148.

    nlp = spacy.load('ru_core_news_sm')
    max_length = nlp.max_length
    with open(file, "r", encoding="utf-8") as f:
        data = f.read()
    chunks = [data[i:i + chunk_size] for i in range(0, len(data), chunk_size)] # Разделяем текст на части, меньше max_length
    chunks = chunks[:max_chunks] # ограничиваем колличество
    results = []
    for idx, chunk in enumerate(chunks): # Итерация с индексами через enumerate
        if len(chunk) > max_length:
            print(f"Пропускаем часть {idx + 1}, так как её длина превышает лимит {max_length} символов.")
            continue

        print(f"Обработка части {idx + 1} из {len(chunks)}...")
        doc = nlp(chunk)  # Обработка текста с помощью spaCy

        # Ищем токены с заданной леммой
        for sent in doc.sents:
            for token in sent:
                if token.lemma_ == s_lemma:
                    results.append(
                        f"Предложение: {sent.text}\n"
                        f"Токен: {token.text}, Лемма: {token.lemma_}, POS: {token.pos_}, Морф: {token.morph}\n"
                    )
    return results



In [19]:
s_lemma = 'толпа'
file = "ru_taiga-ud-train.conllu"
my_result = search_lemma(file, s_lemma)
if my_result:
    print(f"Найдено {len(my_result)} совпадений для леммы '{s_lemma}':\n")
    for result in my_result:
        print(result)
else:
    print(f"Лемма '{s_lemma}' не найдена.")

Обработка части 1 из 10...
Обработка части 2 из 10...
Обработка части 3 из 10...
Обработка части 4 из 10...
Обработка части 5 из 10...
Обработка части 6 из 10...
Обработка части 7 из 10...
Обработка части 8 из 10...
Обработка части 9 из 10...
Обработка части 10 из 10...
Найдено 6 совпадений для леммы 'толпа':

Предложение: PUNCT	_	_	4	punct	_	_

# sent_id = taiga-train-social-2004
# genre = poetry
# text = Император свято блюдет закон, триумфатор пленил врага, Из толпы, из женских рук, из окон — лепестков прозрачных пурга.

Токен: толпы, Лемма: толпа, POS: NOUN, Морф: Animacy=Inan|Case=Gen|Gender=Fem|Number=Sing

Предложение: 1	Император	император	NOUN	_	Animacy=Anim|Case=Nom|Gender=Masc|Number=Sing	3	nsubj	_	_
2	свято	свято	ADV	_	Degree=Pos	3	advmod	_	_
3	блюдет	блюсти	VERB	_	Aspect=Imp|Gender=Masc|Number=Sing|Tense=Past|Variant=Short|VerbForm=Part|Voice=Pass	0	root	_	_
4	закон	закон	NOUN	_	Animacy=Inan|Case=Acc|Gender=Masc|Number=Sing	3	obj	_	SpaceAfter=No
5	,	,	PUNCT	_	_	7	punct	_	_

Поиск по тэгу части речи
Меняем только вводные данные и проверку

In [38]:
def search_pos(file, s_pos, chunk_size=100000, max_chunks=1): # чтобы не занимать много оперативки, обрабатываем по чанкам с размером 100000 символов, и у spacy есть ограничение в 100000 с. 
    #Далее, чтобы не ждать 10 лет, берем толкьо 1 10 чанков из 148.

    nlp = spacy.load('ru_core_news_sm')
    max_length = nlp.max_length
    with open(file, "r", encoding="utf-8") as f:
        data = f.read()
    chunks = [data[i:i + chunk_size] for i in range(0, len(data), chunk_size)] # Разделяем текст на части, меньше max_length
    chunks = chunks[:max_chunks] # ограничиваем колличество
    results = []
    for idx, chunk in enumerate(chunks): # Итерация с индексами через enumerate
        if len(chunk) > max_length:
            print(f"Пропускаем часть {idx + 1}, так как её длина превышает лимит {max_length} символов.")
            continue

        print(f"Обработка части {idx + 1} из {len(chunks)}...")
        doc = nlp(chunk)  # Обработка текста с помощью spaCy

        # Ищем токены с заданным тэгом
        for sent in doc.sents:
            for token in sent:
                if token.pos_ == s_pos:
                    results.append(
                        f"{token.text},{token.pos_}"
                    )
    return results



In [39]:
s_pos = 'ADJ'
file = "ru_taiga-ud-train.conllu"
my_result = search_pos(file, s_pos)
if my_result:
    print(f"Найдено {len(my_result)} совпадений для тэга '{s_pos}':\n")
    for result in my_result:
        print(result)
else:
    print(f"Тэг '{s_pos}' не найден.")

Обработка части 1 из 1...
Найдено 284 совпадений для тэга 'ADJ':

важна,ADJ
важна,ADJ
важный,ADJ
Важно,ADJ
Важно,ADJ
важный,ADJ
больной,ADJ
приятная,ADJ
живой,ADJ
аналитический,ADJ
хорошее,ADJ
сильный,ADJ
небольшая,ADJ
приятная,ADJ
приятный,ADJ
живой,ADJ
живой,ADJ
аналитический,ADJ
аналитический,ADJ
хорошее,ADJ
хороший,ADJ
сильный,ADJ
сильный,ADJ
небольшая,ADJ
небольшой,ADJ
Луганская,ADJ
Донецкая,ADJ
Луганская,ADJ
луганский,ADJ
Донецкая,ADJ
донецкий,ADJ
самом,ADJ
самом,ADJ
самый,ADJ
Левая,ADJ
Левая,ADJ
левый,ADJ
грешном,ADJ
грешном,ADJ
грешный,ADJ
нелюбимая,ADJ
должна,ADJ
должна,ADJ
должен,ADJ
Ясинская,ADJ
2000,ADJ
Ясинская,ADJ
Ясинская,ADJ
2000,ADJ
живыми,ADJ
живыми,ADJ
живой,ADJ
Сильнее,ADJ
сильный,ADJ
Балеевой,ADJ
электронной,ADJ
Балеевой,ADJ
электронной,ADJ
электронный,ADJ
1,ADJ
Милый,ADJ
Милый,ADJ
милый,ADJ
первый,ADJ
первый,ADJ
первый,ADJ
слабее,ADJ
слабее,ADJ
склонны,ADJ
других,ADJ
личностными,ADJ
объективными,ADJ
склонны,ADJ
склонный,ADJ
других,ADJ
другой,ADJ
личностными,ADJ
ли

Поиск по тэгу синтаксической зависимости (DEPS)


В Taiga используются стандартные синтаксические зависимости UD, включая:

Основные синтаксические отношения:

nsubj — подлежащее.

obj — прямое дополнение.

iobj — косвенное дополнение.

root — корень.

Модификаторы:

amod — прилагательное, модифицирующее существительное.

advmod — наречие, модифицирующее глагол, прилагательное или другое наречие.

nummod — числительное, модифицирующее существительное.

Обстоятельства:

obl — обстоятельство.

advcl — придаточное предложение обстоятельства.

Знаки препинания и служебные слова:

punct — знак препинания.

aux — вспомогательный глагол.

cop — связка.

mark — подчинительный союз.

Полный список доступен в документации Universal Dependencies.



In [ ]:
def search_deps(file, s_dep, chunk_size=100000, max_chunks=1): # чтобы не занимать много оперативки, обрабатываем по чанкам с размером 100000 символов, и у spacy есть ограничение в 100000 с. 
    #Далее, чтобы не ждать 10 лет, берем толкьо 1 10 чанков из 148.

    nlp = spacy.load('ru_core_news_sm')
    max_length = nlp.max_length
    with open(file, "r", encoding="utf-8") as f:
        data = f.read()
    chunks = [data[i:i + chunk_size] for i in range(0, len(data), chunk_size)] # Разделяем текст на части, меньше max_length
    chunks = chunks[:max_chunks] # ограничиваем колличество
    results = []
    for idx, chunk in enumerate(chunks): # Итерация с индексами через enumerate
        if len(chunk) > max_length:
            print(f"Пропускаем часть {idx + 1}, так как её длина превышает лимит {max_length} символов.")
            continue

        print(f"Обработка части {idx + 1} из {len(chunks)}...")
        doc = nlp(chunk)  # Обработка текста с помощью spaCy

        # Ищем токены с заданным тэгом
        for sent in doc.sents:
            for token in sent:
                if token.dep_ == s_dep:
                    results.append(
                        f"{sent.text.strip()}, {token.text},{token.dep_}"
                    )
    return results



In [43]:
s_dep = 'advmod'
file = "ru_taiga-ud-train.conllu"
my_result = search_deps(file, s_dep)
if my_result:
    print(f"Найдено {len(my_result)} совпадений для связи '{s_dep}':\n")
    for result in my_result:
        print(result)
else:
    print(f"Связь '{s_dep}' не найдена.")

Обработка части 1 из 1...
Найдено 274 совпадений для связи 'advmod':

# sent_id = taiga-train-social-2
# genre = social
# text = Снова приобрел дозу,
1	Снова	снова	ADV	_	Degree=Pos	2	advmod	_	_
2	приобрел	приобрести	VERB	_	Aspect=Perf|Gender=Masc|Mood=Ind|Number=Sing|Tense=Past|VerbForm=Fin|Voice=Act	0	root	_	_
3	дозу	доза	NOUN	_	Animacy=Inan|Case=Acc|Gender=Fem|Number=Sing	2	obj	_	SpaceAfter=No
4	,	,	PUNCT	_	_	2	punct	_	_

# sent_id = taiga-train-social-4
# genre = social
# text = В женщине важна верность, а не красота., Снова,advmod
# sent_id = taiga-train-social-2
# genre = social
# text = Снова приобрел дозу,
1	Снова	снова	ADV	_	Degree=Pos	2	advmod	_	_
2	приобрел	приобрести	VERB	_	Aspect=Perf|Gender=Masc|Mood=Ind|Number=Sing|Tense=Past|VerbForm=Fin|Voice=Act	0	root	_	_
3	дозу	доза	NOUN	_	Animacy=Inan|Case=Acc|Gender=Fem|Number=Sing	2	obj	_	SpaceAfter=No
4	,	,	PUNCT	_	_	2	punct	_	_

# sent_id = taiga-train-social-4
# genre = social
# text = В женщине важна верность, а не красота., С